# Lecture 30.05: AdaBoost (Adaptive Boosting) — Intuition, Decision Stumps & Sample Reweighting

This notebook contains runnable code aligned with the video lecture demonstration.


In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

def fit_adaboost_classifier_scratch(X, y, n_estimators=4):
    """
    First-principles implementation of Discrete AdaBoost (SAMME) binary classifier.
    Labels must be encoded as {-1, +1}.
    
    Returns:
    --------
    stumps : list
        Trained DecisionTreeClassifier instances of depth 1.
    stump_weights : list
        Calculated alpha_m voting powers.
    final_preds : ndarray
        Ensemble predictions on training set.
    """
    X_arr = np.asarray(X, dtype=float)
    y_arr = np.asarray(y, dtype=float)  # Expected: -1 or +1
    N = len(y_arr)
    
    # 1. Initialize sample weights uniformly: w_i = 1/N
    weights = np.ones(N) / N
    
    stumps = []
    stump_weights = []
    
    for stage in range(n_estimators):
        # Fit decision stump using current sample weights
        stump = DecisionTreeClassifier(max_depth=1, random_state=42 + stage)
        stump.fit(X_arr, y_arr, sample_weight=weights)
        
        preds = stump.predict(X_arr)
        
        # Weighted error eps = sum(w_i * I(y != pred))
        misclassified = (y_arr != preds)
        error = np.sum(weights[misclassified])
        
        # Guard against zero or perfect error
        error = np.clip(error, 1e-10, 1.0 - 1e-10)
        
        # Stump importance alpha = 0.5 * ln((1 - error) / error)
        alpha = 0.5 * np.log((1.0 - error) / error)
        
        # Exponential weight updates
        weights *= np.exp(-alpha * y_arr * preds)
        
        # Normalization
        weights /= np.sum(weights)
        
        stumps.append(stump)
        stump_weights.append(alpha)
        
    # Aggregate predictions across stumps: sign(sum(alpha * h(x)))
    stump_preds = np.array([stump.predict(X_arr) for stump in stumps])
    final_raw = np.dot(np.array(stump_weights), stump_preds)
    final_preds = np.sign(final_raw)
    
    return stumps, stump_weights, final_preds

# ----------------- Driver Execution & Testing -----------------
if __name__ == "__main__":
    # Shraddha ma'am's Blackboard Email Spam Dataset (6 samples)
    # X1: Length, X2: Links, y: Spam (+1) or Not Spam (-1)
    X_emails = np.array([
        [1200.0, 10.0],
        [800.0,  0.0],
        [200.0,  2.0],
        [300.0,  1.0],
        [400.0,  2.0],
        [900.0,  8.0]
    ])
    y_emails = np.array([+1, -1, +1, -1, -1, +1])  # 3 Spam (+1), 3 Ham (-1)
    
    stumps, alphas, preds = fit_adaboost_classifier_scratch(X_emails, y_emails, n_estimators=3)
    
    print("=" * 60)
    print("--- ADABOOST CLASSIFIER (BLACKBOARD SCRATCH DEMO) ---")
    print(f"Stump Importance Weights (Alphas) : {[round(a, 4) for a in alphas]}")
    print(f"Actual Labels                     : {y_emails.tolist()}")
    print(f"AdaBoost Aggregated Predictions   : {[int(p) for p in preds]}")
    print(f"Accuracy                          : {np.mean(preds == y_emails) * 100:.1f}%")
    print("=" * 60)
